In [17]:
%pip install -r requirements.txt

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [18]:
%pip install --force-reinstall "snowflake-connector-python[pandas]"

  Using cached snowflake_connector_python-5.0.0-cp312-cp312-win_amd64.whl.metadata (7.1 kB)
  Using cached cryptography-50.0.2-cp311-abi3-win_amd64.whl.metadata (4.4 kB)
  Using cached protobuf-6.33.6-cp310-abi3-win_amd64.whl.metadata (593 bytes)
  Using cached pytz-2026.5-py2.py3-none-any.whl.metadata (22 kB)
  Using cached pandas-2.3.3-cp312-cp312-win_amd64.whl.metadata (19 kB)
  Using cached pyarrow-26.0.0-cp312-cp312-win_amd64.whl.metadata (3.0 kB)
  Using cached cffi-2.1.1-cp312-cp312-win_amd64.whl.metadata (2.6 kB)
  Using cached numpy-2.5.4-cp312-cp312-win_amd64.whl.metadata (6.6 kB)
  Using cached python_dateutil-2.9.0.post0-py2.py3-none-any.whl.metadata (8.4 kB)
  Using cached tzdata-2026.5-py2.py3-none-any.whl.metadata (1.4 kB)
  Using cached pycparser-3.11-py3-none-any.whl.metadata (8.2 kB)
  Using cached six-1.17.0-py2.py3-none-any.whl.metadata (1.7 kB)
Using cached cryptography-50.0.2-cp311-abi3-win_amd64.whl (3.8 MB)
Using cached pandas-2.3.3-cp312-cp312-win_amd64.whl (11

  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  You can safely remove it manually.
  You can safely remove it manually.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  You can safely remove it manually.
  You can safely remove it manually.

[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import os
from getpass import getpass
from dotenv import load_dotenv
from agents.db import get_connection
from agents.planner import planner_agent
from agents.checker import checker_agent, get_live_alerts
from agents.retriever import search_policies

load_dotenv()

conn = get_connection()
cur = conn.cursor()
MBTA_KEY = os.getenv("MBTA_KEY") or getpass("MBTA key: ").strip()
print("Ready ✅")

In [20]:
print(planner_agent(conn, "Harvard", "Back Bay"))
print(planner_agent(conn, "Harvard", "Back Bay", avoid=["place-dwnxg"]))

{'ok': True, 'path': ['place-harsq', 'place-cntsq', 'place-knncl', 'place-chmnl', 'place-pktrm', 'place-dwnxg', 'place-chncl', 'place-tumnl', 'place-bbsta'], 'stations': ['Harvard', 'Central', 'Kendall/MIT', 'Charles/MGH', 'Park Street', 'Downtown Crossing', 'Chinatown', 'Tufts Medical Center', 'Back Bay'], 'transfers': ['Downtown Crossing'], 'transfer_ids': ['place-dwnxg']}
{'ok': False, 'reason': 'No route found avoiding those stations'}


In [21]:
print(planner_agent(conn, "Harvard", "Haymarket")["stations"])
print(planner_agent(conn, "Harvard", "Haymarket", avoid=["place-gover"])["stations"])

['Harvard', 'Central', 'Kendall/MIT', 'Charles/MGH', 'Park Street', 'Government Center', 'Haymarket']
['Harvard', 'Central', 'Kendall/MIT', 'Charles/MGH', 'Park Street', 'Downtown Crossing', 'State', 'Haymarket']


In [22]:
tests = [
    # (origin, destination, avoid, what you should see)
    ("Harvard", "Back Bay", None, "Red to Downtown Crossing, then Orange"),
    ("Wonderland", "Forest Hills", None, "Blue to State, then Orange"),
    ("Alewife", "Ashmont", None, "Red Line only, no transfers"),
    ("Ashmont", "Braintree", None, "must go through JFK/UMass (branch fix)"),
    ("Harvard", "Haymarket", ["place-gover"], "avoids Government Center"),
    ("harvard", "KENDALL/MIT", None, "works with any capitalization"),
    ("Hogwarts", "Harvard", None, "should fail: station not found"),
    ("Boylston", "Harvard", None, "should fail: not wheelchair accessible"),
]

for origin, dest, avoid, expected in tests:
    result = planner_agent(conn, origin, dest, avoid=avoid)
    print(f"{origin} -> {dest}" + (f"  (avoid {avoid})" if avoid else ""))
    print(f"  Expected: {expected}")
    if result["ok"]:
        print(f"  Route:     {' -> '.join(result['stations'])}")
        print(f"  Transfers: {result['transfers'] or 'none'}")
    else:
        print(f"  Failed:    {result['reason']}")
    print()

Harvard -> Back Bay
  Expected: Red to Downtown Crossing, then Orange
  Route:     Harvard -> Central -> Kendall/MIT -> Charles/MGH -> Park Street -> Downtown Crossing -> Chinatown -> Tufts Medical Center -> Back Bay
  Transfers: ['Downtown Crossing']

Wonderland -> Forest Hills
  Expected: Blue to State, then Orange
  Route:     Wonderland -> Revere Beach -> Beachmont -> Suffolk Downs -> Orient Heights -> Wood Island -> Airport -> Maverick -> Aquarium -> State -> Downtown Crossing -> Chinatown -> Tufts Medical Center -> Back Bay -> Massachusetts Avenue -> Ruggles -> Roxbury Crossing -> Jackson Square -> Stony Brook -> Green Street -> Forest Hills
  Transfers: ['State']

Alewife -> Ashmont
  Expected: Red Line only, no transfers
  Route:     Alewife -> Davis -> Porter -> Harvard -> Central -> Kendall/MIT -> Charles/MGH -> Park Street -> Downtown Crossing -> South Station -> Broadway -> Andrew -> JFK/UMass -> Savin Hill -> Fields Corner -> Shawmut -> Ashmont
  Transfers: none

Ashmont -

In [23]:
for a in get_live_alerts(MBTA_KEY)[:10]:
    print("-", a["attributes"]["effect"], "|", a["attributes"]["header"][:100])

- ESCALATOR_CLOSURE | Courthouse Escalator 417 (Airport/Design Center platform to south lobby) is unavailable due to ongoi
- ESCALATOR_CLOSURE | State Escalator 6 (Oak Grove platform to Old State House lobby) unavailable
- ACCESS_ISSUE | The Central Avenue mini-high platform (Ashmont side) is out of service. 
- ELEVATOR_CLOSURE | Back Bay Elevator 854 (Commuter Rail track 2 to lobby) is unavailable.
- ELEVATOR_CLOSURE | Worcester Elevator 782 (Parking lot to pedestrian bridge) is unavailable. Please use elevator in mai
- ESCALATOR_CLOSURE | Bowdoin Escalator 125 (Lobby to street) is unavailable until further notice, pending replacement.
- ESCALATOR_CLOSURE | South Station Escalator 419 (Lobby to South Station platform) is unavailable due to maintenance.
- ESCALATOR_CLOSURE | Courthouse Escalator 418 (Lobby to Seaport Boulevard, Fan Pier Boulevard) is unavailable due to main
- ESCALATOR_CLOSURE | Malden Center Escalator 312 (Street level to Commuter Rail platform) unavailable due to mai

In [24]:
plan = planner_agent(conn, "Harvard", "Back Bay")
check = checker_agent(plan, MBTA_KEY)

print("Route:", " → ".join(plan["stations"]))
print("Safe to travel:", check["safe"])
for p in check["all_issues"]:
    print(f"  [{p['role']}] {p['station']}: {p['effect']} — {p['alert'][:80]}")

Route: Harvard → Central → Kendall/MIT → Charles/MGH → Park Street → Downtown Crossing → Chinatown → Tufts Medical Center → Back Bay
Safe to travel: False
  [destination] Back Bay: ELEVATOR_CLOSURE — Back Bay Elevator 854 (Commuter Rail track 2 to lobby) is unavailable.
  [transfer] Downtown Crossing: ELEVATOR_CLOSURE — Downtown Crossing Elevator 869 (Alewife platform to Summer Street Concourse) is 


In [25]:
for p in check["blocking"]:
    print(p["station"], "→", p["alert"])

Back Bay → Back Bay Elevator 854 (Commuter Rail track 2 to lobby) is unavailable.
Downtown Crossing → Downtown Crossing Elevator 869 (Alewife platform to Summer Street Concourse) is unavailable due to maintenance through Tuesday, December 1.


In [26]:
print(planner_agent(conn, "Ashmont", "Braintree")["stations"])

['Ashmont', 'Shawmut', 'Fields Corner', 'Savin Hill', 'JFK/UMass', 'North Quincy', 'Wollaston', 'Quincy Center', 'Quincy Adams', 'Braintree']


In [27]:
from agents.replanner import replanner_agent

result = replanner_agent(conn, "Harvard", "Back Bay", MBTA_KEY)

if not result["ok"]:
    print("Failed:", result["reason"])
elif result["changed"]:
    print("Original:", " → ".join(result["original_plan"]["stations"]))
    print("Problems:", [p["station"] for p in result["problems"]])
    print("New route:", " → ".join(result["new_plan"]["stations"]) if result["new_plan"] else "none found")
    print("\nExplanation:\n", result["explanation"])
    print("\nSources:", result["sources"])
else:
    print(result["explanation"])

Original: Harvard → Central → Kendall/MIT → Charles/MGH → Park Street → Downtown Crossing → Chinatown → Tufts Medical Center → Back Bay
Problems: ['Back Bay', 'Downtown Crossing']
New route: none found

Explanation:
 **Your route has step-free barriers, and no fully step-free alternative was found.**

- **Back Bay (destination):** Elevator 854, from Commuter Rail track 2 to the lobby, is unavailable [mbta_elevator_alerts.pdf].
- **Downtown Crossing (transfer):** Elevator 869, from the Alewife platform to the Summer Street Concourse, is out for maintenance through Tuesday, December 1 [station_downtown_crossing.pdf].
- **Kendall/MIT:** Elevator 777, from the Alewife platform to Main Street, is also unavailable [mbta_elevator_alerts.pdf]. Your route passes through Kendall, so check this stop too.

**Recommendation:** I can't identify a safe step-free option from the information available, so I can't recommend this route as planned. Contact MBTA customer support at 617-222-3200 (TTY: 711) 

In [28]:
from agents.monitor import start_trip, check_trip, monitor_trip

trip = start_trip(conn, "Harvard", "Back Bay", MBTA_KEY)
print("Route:", " → ".join(trip["plan"]["stations"]))
print("Outages already known:", trip["known_alerts"])

Route: Harvard → Central → Kendall/MIT → Charles/MGH → Park Street → Downtown Crossing → Chinatown → Tufts Medical Center → Back Bay
Outages already known: {'1031518', '1038229'}


In [29]:
event = check_trip(conn, trip, MBTA_KEY)
print(event["time"], "| new problems:", len(event["new_problems"]), "| resolved:", event["resolved"])

20:02:52 | new problems: 0 | resolved: []


In [30]:
trip["known_alerts"] = set()   # pretend these outages just appeared

event = check_trip(conn, trip, MBTA_KEY)
print("New problems:", [p["station"] for p in event["new_problems"]])
if event["replan"]:
    print("\nReplanner says:\n", event["replan"]["explanation"])

New problems: ['Back Bay', 'Downtown Crossing']

Replanner says:
 **Problem:** Your route has two step-free barriers. At Downtown Crossing, Elevator 869 (Alewife platform to Summer Street Concourse) is out for maintenance through Tuesday, December 1 [station_downtown_crossing.pdf]. At Back Bay, Elevator 854 (Commuter Rail track 2 to lobby) is unavailable [mbta_elevator_alerts.pdf].

**Recommended option:** Our planner found no fully step-free route for this trip, and the MBTA sources don't name a replacement elevator that completes it, so I can't recommend a specific route.

**What to do:**
- **Call MBTA customer support at 617-222-3200** (TTY: 711), Monday to Friday, 6:30 AM to 8 PM [station_downtown_crossing.pdf]. Ask them to confirm a step-free path through Downtown Crossing and Back Bay.
- **Check the Back Bay alert's scope.** It names Commuter Rail track 2, so confirm which elevators serve the platform you'll use [mbta_elevator_alerts.pdf].
- **Ask station staff** at Downtown Cros

In [31]:
def show(event):
    print(f"[{event['time']}] new: {len(event['new_problems'])}, resolved: {len(event['resolved'])}",
          "→ REPLANNED" if event["replan"] else "")

monitor_trip(conn, trip, MBTA_KEY, interval_sec=10, max_checks=3, on_event=show)

[20:03:02] new: 0, resolved: 0 
[20:03:13] new: 0, resolved: 0 
[20:03:24] new: 0, resolved: 0 


[{'time': '20:02:52', 'new_problems': [], 'resolved': [], 'replan': None},
 {'time': '20:02:53',
  'new_problems': [{'station_id': 'place-bbsta',
    'station': 'Back Bay',
    'role': 'destination',
    'effect': 'ELEVATOR_CLOSURE',
    'alert': 'Back Bay Elevator 854 (Commuter Rail track 2 to lobby) is unavailable.',
    'alert_id': '1038229'},
   {'station_id': 'place-dwnxg',
    'station': 'Downtown Crossing',
    'role': 'transfer',
    'effect': 'ELEVATOR_CLOSURE',
    'alert': 'Downtown Crossing Elevator 869 (Alewife platform to Summer Street Concourse) is unavailable due to maintenance through Tuesday, December 1.',
    'alert_id': '1031518'}],
  'resolved': [],
  'replan': {'ok': True,
   'changed': True,
   'original_plan': {'ok': True,
    'path': ['place-harsq',
     'place-cntsq',
     'place-knncl',
     'place-chmnl',
     'place-pktrm',
     'place-dwnxg',
     'place-chncl',
     'place-tumnl',
     'place-bbsta'],
    'stations': ['Harvard',
     'Central',
     'Kend

In [35]:
from agents.retriever import search_policies

q = "Elevator 869 is closed at Downtown Crossing. What should I do?"

print("VECTOR ONLY:")
for r in search_policies(conn, q, keyword_boost=0):
    print(r["score"], "|", r["source"], "|", r["chunk"][:90])

print("\nHYBRID:")
for r in search_policies(conn, q):
    print(r["score"], f"(vector {r['vector_score']} + keyword {r['keyword_boost']})", "|", r["source"], "|", r["chunk"][:90])

VECTOR ONLY:


TypeError: search_policies() got an unexpected keyword argument 'keyword_boost'